In [ ]:
import csv

data = []

# 1. Open and read the file
with open('data.csv', mode='r', encoding='utf-8') as f:
    # Use csv.reader to handle the "ID","Joke" structure
    reader = csv.reader(f)
    
    # Skip the header row ("ID", "Joke")
    next(reader)
    
    # Loop through each row and grab the second column (index 1)
    for row in reader:
        if len(row) > 1:
            data.append(row[1])

In [ ]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments

# 1. Load Tokenizer and Model
model = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model)
tokenizer.pad_token = tokenizer.eos_token # GPT2 doesn't have a pad token by default

# 2. Format your 1,622 jokes
# Each joke should look like: "<|startoftext|> Why did the... <|endoftext|>"
formatted_jokes = [f"<|startoftext|> {joke} <|endoftext|>" for joke in data]

In [ ]:
print(f"Successfully loaded {len(data)} jokes.")
print(f"Example: {formatted_jokes[0]}")

In [ ]:
# from transformers import GPT2LMHeadModel, GPT2Tokenizer
# import torch

# # Load tools
# tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
# tokenizer.pad_token = tokenizer.eos_token
# model = GPT2LMHeadModel.from_pretrained("gpt2")
# device = "cpu"
# model.to(device)
# model.eval() # Set to evaluation mode



In [ ]:
# def generate_joke(seed_words, model, tokenizer):
#     # This creates both 'input_ids' and the 'attention_mask'
#     inputs = tokenizer(seed_words, return_tensors='pt', padding=True).to(device)
    
#     with torch.no_grad(): # This prevents memory-heavy math tracking
#         output_sequences = model.generate(
#             input_ids=inputs['input_ids'],
#             attention_mask=inputs['attention_mask'],
#             max_new_tokens=15,
#             temperature=0.8,
#             top_k=50,
#             top_p=0.95,
#             do_sample=True,
#             pad_token_id=tokenizer.eos_token_id
#         )
    
#     return tokenizer.decode(output_sequences[0], skip_special_tokens=True)

# # Define your seeds
# dataset_seeds = ["What did the", "Why did the", "I went for"]
# random_seeds = ["The purple toaster", "Running is actually", "Under the ocean"]

# # Execute
# print("--- RESULTS ---")
# for seed in dataset_seeds + random_seeds:
#     print(f"Seed: {seed}")
#     print(f"Joke: {generate_joke(seed, model, tokenizer)}")
#     print("-" * 20)

In [ ]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# 1. Load objects
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2") # Or your local folder path

# 2. Freeze the model immediately
model.eval()
device = "cpu"
model.to(device)

# 3. Define a super-light generation function
def generate_safe(seed):
    # Truncation ensures the input isn't too long
    inputs = tokenizer(seed, return_tensors='pt', truncation=True, max_length=10).to(device)
    
    with torch.no_grad():
        output_ids = model.generate(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            max_new_tokens=20,     # Very short generation to save RAM
            do_sample=True,
            temperature=0.7,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

# 4. Run seeds one-by-one (don't use a loop yet, just to test)
print(f"Dataset Seed: {generate_safe('What did the')}")
print(f"Random Seed: {generate_safe('The purple toaster')}")